In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'all-MiniLM-L6-v2'
LOCAL_MODELS_FOLDER = './models/'

local_path = os.path.join(LOCAL_MODELS_FOLDER, MODEL_NAME)

if not os.path.isdir(local_path):
    model = SentenceTransformer(MODEL_NAME)
    os.makedirs(local_path, exist_ok=True)
    model.save(local_path)

model = SentenceTransformer(local_path)

/workspaces/llm-zoomcamp-2026/Week_2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 808.59it/s]


In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
v1.dot(dv)

np.float32(0.32332397)

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
v2.dot(dv)

np.float32(0.019730438)

In [8]:
from ingest import load_faq_data

documents = load_faq_data()

In [9]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [10]:
from tqdm.auto import tqdm

In [11]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

100%|██████████| 27/27 [01:54<00:00,  4.23s/it]


1350

In [12]:
vectors[10].shape

(384,)

In [13]:
import numpy as np

X = np.array(vectors)

In [14]:
scores = X.dot(v1)

In [15]:
top5 = np.argsort(-scores)[:5]
top5


array([  2, 625, 907, 538,   7])

In [16]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192132
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [17]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [18]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [25]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

vs_index.fit(vectors, documents)

In [29]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
    )

In [31]:
vs_index.close()